In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

ACCENT = '#6c63ff'
GREEN  = '#00d4aa'
PINK   = '#ff6b9d'
ORANGE = '#ffb347'
BLUE   = '#4fc3f7'
TEXT   = '#e0e0e0'
AC = {'Walking': BLUE, 'Running': PINK, 'Resting': GREEN, 'Cycling': ORANGE}
PT = dict(layout=dict(
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(26,29,46,0.8)',
    font=dict(color=TEXT, family='Inter,sans-serif'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.08)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.08)'),
    margin=dict(l=40, r=20, t=50, b=40),
    legend=dict(bgcolor='rgba(26,29,46,0.7)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1)
))

In [ ]:
np.random.seed(42)
N = 1000
acts = np.random.choice(['Walking','Running','Resting','Cycling'], N, p=[0.30,0.25,0.25,0.20])
HRP = {'Walking':(95,15),   'Running':(160,20), 'Resting':(65,10),  'Cycling':(130,18)}
STP = {'Walking':(5500,1200),'Running':(9000,1500),'Resting':(300,150),'Cycling':(4000,800)}
CAP = {'Walking':(280,60),  'Running':(600,80),  'Resting':(60,15),  'Cycling':(450,70)}
SPP = {'Walking':(97.5,1.0),'Running':(96.0,1.5),'Resting':(98.5,0.5),'Cycling':(97.0,1.2)}
hr = [max(45, min(220, np.random.normal(*HRP[a]))) for a in acts]
st = [max(0,  int(np.random.normal(*STP[a])))      for a in acts]
ca = [max(20, np.random.normal(*CAP[a]))            for a in acts]
sp = [min(100,max(88, np.random.normal(*SPP[a])))   for a in acts]
df = pd.DataFrame({
    'Activity_Status': acts,
    'Heart_Rate':      np.round(hr, 1),
    'Step_Count':      st,
    'Calories_Burned': np.round(ca, 1),
    'SpO2':            np.round(sp, 1),
    'Age':             np.random.randint(18, 65, N),
    'Athlete_Level':   np.random.choice(['Beginner','Intermediate','Advanced'], N, p=[0.4,0.35,0.25])
})
print(f'Dataset listo: {df.shape[0]} registros x {df.shape[1]} variables')

In [ ]:
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
body{background:#0f1117!important}
.dbh{background:linear-gradient(135deg,#1a1d2e,#0f1117);border:1px solid rgba(108,99,255,.3);
     border-radius:16px;padding:28px 36px;margin-bottom:20px;position:relative;overflow:hidden}
.dbh::before{content:'';position:absolute;top:0;left:0;right:0;height:3px;
             background:linear-gradient(90deg,#6c63ff,#00d4aa,#ff6b9d)}
.dbt{font-family:Inter,sans-serif;font-size:24px;font-weight:700;color:#fff;margin:0 0 6px}
.dbs{font-family:Inter,sans-serif;font-size:13px;color:#9ca3af}
.badge{display:inline-block;background:rgba(108,99,255,.2);color:#6c63ff;
       border:1px solid rgba(108,99,255,.4);border-radius:20px;
       padding:3px 12px;font-size:11px;font-weight:600;margin:8px 4px 0 0}
.kpi-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:20px 0}
.kpi{background:#1a1d2e;border:1px solid rgba(255,255,255,.08);border-radius:12px;padding:18px 20px}
.kl{font-family:Inter,sans-serif;font-size:11px;color:#9ca3af;
    text-transform:uppercase;letter-spacing:.08em;margin-bottom:6px}
.kv{font-family:Inter,sans-serif;font-size:26px;font-weight:700;color:#fff}
.kd{font-family:Inter,sans-serif;font-size:11px;margin-top:3px}
.st{font-family:Inter,sans-serif;font-size:15px;font-weight:600;color:#e0e0e0;
    margin:20px 0 10px;padding-left:10px;border-left:3px solid #6c63ff}
</style>
<div class='dbh'>
  <div class='dbt'>Dashboard de Salud Deportiva con Wearables</div>
  <div class='dbs'>Analisis interactivo | Ciencia de Datos II | 2026</div>
  <span class='badge'>Analisis Exploratorio</span>
  <span class='badge'>Regresion Lineal</span>
  <span class='badge'>Anomalias IQR</span>
  <span class='badge'>Segmentacion por Actividad</span>
</div>
"""))

In [ ]:
sl = {'description_width': '130px'}
af  = widgets.SelectMultiple(
        options=['Walking','Running','Resting','Cycling'],
        value=['Walking','Running','Resting','Cycling'],
        description='Actividades:', style=sl,
        layout=widgets.Layout(width='280px'))
hrs = widgets.IntRangeSlider(
        value=[40,220], min=40, max=220, step=5,
        description='FC range:', style=sl,
        layout=widgets.Layout(width='380px'))
alf = widgets.RadioButtons(
        options=['Todos','Beginner','Intermediate','Advanced'],
        value='Todos', description='Nivel:', style=sl)
ct  = widgets.ToggleButtons(
        options=['Distribucion','Correlacion','Regresion','Boxplot'],
        description='Vista:',
        style={'description_width':'60px','button_width':'110px'})
cb  = widgets.VBox(
        [widgets.HTML('<div style="font-family:Inter;font-size:12px;font-weight:600;'
                      'color:#9ca3af;text-transform:uppercase;letter-spacing:.08em;'
                      'margin-bottom:8px">CONTROLES</div>'),
         widgets.HBox([af, widgets.VBox([hrs, alf])]), ct],
        layout=widgets.Layout(
            background='#1a1d2e', padding='18px',
            margin='0 0 18px 0',
            border='1px solid rgba(255,255,255,.08)'))
display(cb)

In [ ]:
ko = widgets.Output()
mo = widgets.Output()
bo = widgets.Output()
display(ko)
display(mo)
display(bo)


def gf(at, hr, al):
    m = (df.Activity_Status.isin(list(at)) &
         (df.Heart_Rate >= hr[0]) & (df.Heart_Rate <= hr[1]))
    if al != 'Todos':
        m &= (df.Athlete_Level == al)
    return df[m].copy()


def rk(f):
    n   = len(f)
    ah  = f.Heart_Rate.mean()
    as_ = f.Step_Count.mean()
    cr  = f[['Step_Count','Heart_Rate']].corr().iloc[0,1]
    pn  = n / len(df) * 100
    cc  = '#00d4aa' if abs(cr) > 0.5 else '#ffb347'
    cl  = 'Alta' if abs(cr) > 0.5 else 'Baja'
    return (
        f"<div class='kpi-grid'>"
        f"<div class='kpi' style='border-top:3px solid {ACCENT}'>"
        f"<div class='kl'>Registros</div><div class='kv'>{n:,}</div>"
        f"<div class='kd' style='color:#9ca3af'>{pn:.1f}% del total</div></div>"
        f"<div class='kpi' style='border-top:3px solid {PINK}'>"
        f"<div class='kl'>FC Media</div>"
        f"<div class='kv' style='color:{PINK}'>{ah:.1f} bpm</div></div>"
        f"<div class='kpi' style='border-top:3px solid {BLUE}'>"
        f"<div class='kl'>Pasos Promedio</div>"
        f"<div class='kv' style='color:{BLUE}'>{as_:,.0f}</div></div>"
        f"<div class='kpi' style='border-top:3px solid {GREEN}'>"
        f"<div class='kl'>Correlacion Pasos-FC</div>"
        f"<div class='kv' style='color:{GREEN}'>{cr:+.3f}</div>"
        f"<div class='kd' style='color:{cc}'>{cl}</div></div></div>"
    )


def md(f):
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Distribucion FC', 'Distribucion Pasos'])
    for a in f.Activity_Status.unique():
        s = f[f.Activity_Status == a]
        c = AC.get(a, '#fff')
        fig.add_trace(go.Histogram(x=s.Heart_Rate, name=a,
                                   marker_color=c, opacity=0.7), row=1, col=1)
        fig.add_trace(go.Histogram(x=s.Step_Count, name=a, showlegend=False,
                                   marker_color=c, opacity=0.7), row=1, col=2)
    fig.update_layout(**PT['layout'], height=380, barmode='overlay')
    return fig


def mc(f):
    fig = go.Figure()
    for a in f.Activity_Status.unique():
        s = f[f.Activity_Status == a]
        c = AC.get(a, '#fff')
        fig.add_trace(go.Scatter(
            x=s.Step_Count, y=s.Heart_Rate, mode='markers', name=a,
            marker=dict(color=c, size=6, opacity=0.6)))
    fig.update_layout(**PT['layout'], height=380,
                      xaxis_title='Pasos', yaxis_title='FC (bpm)',
                      title='Correlacion: Pasos vs FC')
    return fig


def mr(f):
    fig = go.Figure()
    x = f.Step_Count.values.astype(float)
    y = f.Heart_Rate.values.astype(float)
    for a in f.Activity_Status.unique():
        s = f[f.Activity_Status == a]
        c = AC.get(a, '#fff')
        fig.add_trace(go.Scatter(
            x=s.Step_Count, y=s.Heart_Rate, mode='markers', name=a,
            marker=dict(color=c, size=6, opacity=0.5)))
    if len(x) > 1:
        cs = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 200)
        yp = np.polyval(cs, x)
        r2 = 1 - np.sum((y - yp)**2) / np.sum((y - y.mean())**2)
        fig.add_trace(go.Scatter(
            x=xl, y=np.polyval(cs, xl), mode='lines',
            name=f'Regresion (R2={r2:.3f})',
            line=dict(color='#ff4444', width=2.5)))
        fig.add_annotation(
            x=0.05, y=0.95, xref='paper', yref='paper',
            text=f'y = {cs[0]:.4f}x + {cs[1]:.2f}<br>R2 = {r2:.4f}',
            showarrow=False, bgcolor='rgba(26,29,46,0.85)',
            bordercolor=ACCENT, borderwidth=1,
            font=dict(color='white', size=12), align='left')
    fig.update_layout(**PT['layout'], height=380,
                      xaxis_title='Pasos', yaxis_title='FC (bpm)',
                      title='Regresion Lineal: Pasos -> FC')
    return fig


def mb(f):
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['FC por Actividad', 'Calorias por Actividad'])
    for a in ['Walking', 'Running', 'Resting', 'Cycling']:
        s = f[f.Activity_Status == a]
        if len(s) == 0:
            continue
        c = AC.get(a, '#fff')
        fig.add_trace(go.Box(y=s.Heart_Rate, name=a,
                             marker_color=c, boxmean=True), row=1, col=1)
        fig.add_trace(go.Box(y=s.Calories_Burned, name=a, showlegend=False,
                             marker_color=c, boxmean=True), row=1, col=2)
    fig.update_layout(**PT['layout'], height=380)
    return fig


def raf(f):
    cats = ['Heart_Rate','Step_Count','Calories_Burned','SpO2','Age']
    lbls = ['FC','Pasos','Calorias','SpO2','Edad']
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type':'polar'}, {'type':'xy'}]],
        subplot_titles=['Perfil Fisiologico', 'Anomalias IQR'])
    gm = f.groupby('Activity_Status')[cats].mean()
    nm = (gm - gm.min()) / (gm.max() - gm.min() + 1e-9)
    for a, r in nm.iterrows():
        c = AC.get(a, '#fff')
        v = list(r.values) + [r.values[0]]
        l = lbls + [lbls[0]]
        fig.add_trace(go.Scatterpolar(
            r=v, theta=l, fill='toself', name=a, line_color=c), row=1, col=1)
    od = []
    for col in ['Heart_Rate', 'Step_Count', 'Calories_Burned']:
        q1, q3 = f[col].quantile([0.25, 0.75])
        iq = q3 - q1
        no = ((f[col] < q1-1.5*iq) | (f[col] > q3+1.5*iq)).sum()
        od.append({'Variable': col.replace('_',' '),
                   'Atipicos': no, 'Pct': no/len(f)*100})
    odf = pd.DataFrame(od)
    fig.add_trace(go.Bar(
        x=odf.Variable, y=odf.Atipicos,
        marker_color=[PINK, BLUE, ORANGE],
        text=[f'{p:.1f}%' for p in odf.Pct],
        textposition='outside', showlegend=False), row=1, col=2)
    fig.update_layout(**PT['layout'], height=360)
    fig.update_polars(
        bgcolor='rgba(26,29,46,0.8)',
        radialaxis=dict(visible=True, gridcolor='rgba(255,255,255,0.1)'),
        angularaxis=dict(gridcolor='rgba(255,255,255,0.1)'))
    return fig


CHS = {'Distribucion': md, 'Correlacion': mc, 'Regresion': mr, 'Boxplot': mb}


def upd(at, hr, al, V):
    f = gf(at, hr, al)
    with ko:
        ko.clear_output(wait=True)
        if len(f) > 0:
            display(HTML(rk(f)))
        else:
            display(HTML('<p style="color:#ff6b9d">Sin datos con los filtros actuales.</p>'))
    with mo:
        mo.clear_output(wait=True)
        display(HTML(f'<div class="st">Vista: {V}</div>'))
        CHS[V](f).show(config={'displayModeBar': True, 'displaylogo': False})
    with bo:
        bo.clear_output(wait=True)
        display(HTML('<div class="st">Perfil Fisiologico y Anomalias</div>'))
        raf(f).show(config={'displayModeBar': False})


out = widgets.interactive_output(upd, {'at': af, 'hr': hrs, 'al': alf, 'V': ct})
display(out)

In [ ]:
display(HTML("""
<div style='background:#1a1d2e;border:1px solid rgba(255,255,255,.06);
     border-radius:12px;padding:16px 22px;margin-top:18px;
     font-family:Inter,sans-serif;font-size:12px;color:#9ca3af'>
  <strong style='color:#6c63ff'>Fundacion Universitaria Compensar</strong>
  &middot; Programacion para Ciencia de Datos II &middot; 2026-1
  &middot; Alvaro Enrique Ospino Mantilla &amp; Daylher Fabian Rodriguez Pena<br>
  <span style='color:#4fc3f7'>Stack:</span>
  Python &middot; NumPy &middot; Pandas &middot; Plotly &middot; ipywidgets &middot; Voila
</div>
"""))